<a href="https://colab.research.google.com/github/manveethjain8/Machine_Learning_II/blob/main/FOIL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import pandas as pd

df = pd.read_csv("/content/zoo_relational_compact.csv")

df.head()

,animal,predicate,value,type
0,aardvark,hair,true,1
1,aardvark,milk,true,1
2,aardvark,predator,true,1
3,aardvark,toothed,true,1
4,aardvark,backbone,true,1


In [21]:
# Check how many animals belong to each class
df.groupby("type")["animal"].nunique()


,animal
type,
1,41
2,20
3,5
4,13
5,4
6,8
7,10


In [22]:
class_names = {
    1: "mammal",
    2: "bird",
    3: "reptile",
    4: "fish",
    5: "amphibian",
    6: "insect",
    7: "invertebrate"
}

df["class_name"] = df["type"].map(class_names)

df[["animal", "class_name"]].drop_duplicates().head(20)

,animal,class_name
0,aardvark,mammal
8,antelope,mammal
16,bass,fish
24,bear,mammal
32,boar,mammal
41,buffalo,mammal
49,calf,mammal
58,carp,fish
66,catfish,fish
74,cavy,mammal


In [23]:
def row_to_fact(row):
    animal = row["animal"]
    predicate = row["predicate"]
    value = row["value"]

    if predicate == "legs":
        return f"{predicate}({animal},{value})"
    else:
        return f"{predicate}({animal})"


# Convert every row into a logical fact
df["fact"] = df.apply(row_to_fact, axis=1)

df[["animal", "predicate", "value", "fact"]].head(15)

,animal,predicate,value,fact
0,aardvark,hair,true,hair(aardvark)
1,aardvark,milk,true,milk(aardvark)
2,aardvark,predator,true,predator(aardvark)
3,aardvark,toothed,true,toothed(aardvark)
4,aardvark,backbone,true,backbone(aardvark)
5,aardvark,breathes,true,breathes(aardvark)
6,aardvark,catsize,true,catsize(aardvark)
7,aardvark,legs,4,"legs(aardvark,4)"
8,antelope,hair,true,hair(antelope)
9,antelope,milk,true,milk(antelope)


In [24]:
background_knowledge = {}

for animal, group in df.groupby("animal"):
    background_knowledge[animal] = set(group["fact"])


# Look at one animal
print("Background knowledge for aardvark:")
for fact in sorted(background_knowledge["aardvark"]):
    print(" ", fact)

Background knowledge for aardvark:
  backbone(aardvark)
  breathes(aardvark)
  catsize(aardvark)
  hair(aardvark)
  legs(aardvark,4)
  milk(aardvark)
  predator(aardvark)
  toothed(aardvark)


In [25]:
# Positive examples for the mammal class
positive_examples = set(
    df[df["class_name"] == "mammal"]["animal"]
)

print("Number of positive examples:", len(positive_examples))
print("\nPositive examples:")
print(sorted(positive_examples))

Number of positive examples: 41

Positive examples:
['aardvark', 'antelope', 'bear', 'boar', 'buffalo', 'calf', 'cavy', 'cheetah', 'deer', 'dolphin', 'elephant', 'fruitbat', 'giraffe', 'girl', 'goat', 'gorilla', 'hamster', 'hare', 'leopard', 'lion', 'lynx', 'mink', 'mole', 'mongoose', 'opossum', 'oryx', 'platypus', 'polecat', 'pony', 'porpoise', 'puma', 'pussycat', 'raccoon', 'reindeer', 'seal', 'sealion', 'squirrel', 'vampire', 'vole', 'wallaby', 'wolf']


In [26]:
all_animals = set(df["animal"])

negative_examples = all_animals - positive_examples

print("Number of negative examples:", len(negative_examples))
print("\nFirst 20 negative examples:")
print(sorted(negative_examples)[:20])

Number of negative examples: 60

First 20 negative examples:
['bass', 'carp', 'catfish', 'chicken', 'chub', 'clam', 'crab', 'crayfish', 'crow', 'dogfish', 'dove', 'duck', 'flamingo', 'flea', 'frog', 'frog_2', 'gnat', 'gull', 'haddock', 'hawk']


In [27]:
predicates = sorted(df["predicate"].unique())

print("Predicates:")
for p in predicates:
    print(p)

Predicates:
airborne
aquatic
backbone
breathes
catsize
domestic
eggs
feathers
fins
hair
legs
milk
predator
tail
toothed
venomous


In [28]:
candidate_rules = []

for predicate in predicates:
    if predicate == "legs":
        # legs needs a value, so skip it for now
        continue

    rule = f"mammal(X) :- {predicate}(X)."
    candidate_rules.append(rule)

print("Candidate rules:")
for rule in candidate_rules:
    print(rule)

Candidate rules:
mammal(X) :- airborne(X).
mammal(X) :- aquatic(X).
mammal(X) :- backbone(X).
mammal(X) :- breathes(X).
mammal(X) :- catsize(X).
mammal(X) :- domestic(X).
mammal(X) :- eggs(X).
mammal(X) :- feathers(X).
mammal(X) :- fins(X).
mammal(X) :- hair(X).
mammal(X) :- milk(X).
mammal(X) :- predator(X).
mammal(X) :- tail(X).
mammal(X) :- toothed(X).
mammal(X) :- venomous(X).


In [29]:
def covers(animal, predicate):
    fact = f"{predicate}({animal})"
    return fact in background_knowledge[animal]

In [31]:
print(covers("lion", "hair"))
print(covers("lion", "feathers"))

print(covers("chicken", "hair"))
print(covers("chicken", "feathers"))

True
False
False
True


In [32]:
def evaluate_rule(predicate):
    true_positives = 0
    false_positives = 0

    for animal in positive_examples:
        if covers(animal, predicate):
            true_positives += 1

    for animal in negative_examples:
        if covers(animal, predicate):
            false_positives += 1

    return true_positives, false_positives


tp, fp = evaluate_rule("hair")

print("True positives :", tp)
print("False positives:", fp)

True positives : 39
False positives: 4


In [33]:
for predicate in predicates:
    if predicate != "legs":
        tp, fp = evaluate_rule(predicate)
        print(f"{predicate:10} TP={tp:2} FP={fp:2}")

airborne   TP= 2 FP=22
aquatic    TP= 6 FP=30
backbone   TP=41 FP=42
breathes   TP=41 FP=39
catsize    TP=32 FP=12
domestic   TP= 8 FP= 5
eggs       TP= 1 FP=58
feathers   TP= 0 FP=20
fins       TP= 4 FP=13
hair       TP=39 FP= 4
milk       TP=41 FP= 0
predator   TP=22 FP=34
tail       TP=35 FP=40
toothed    TP=40 FP=21
venomous   TP= 0 FP= 8


In [34]:
def score_rule(predicate):
    tp, fp = evaluate_rule(predicate)

    if tp + fp == 0:
        precision = 0
    else:
        precision = tp / (tp + fp)

    return tp, fp, precision

In [35]:
rule_scores = []

for predicate in predicates:
    if predicate == "legs":
        continue

    tp, fp, precision = score_rule(predicate)

    rule_scores.append({
        "predicate": predicate,
        "tp": tp,
        "fp": fp,
        "precision": precision
    })

scores_df = pd.DataFrame(rule_scores)

scores_df = scores_df.sort_values(
    by="precision",
    ascending=False
)

scores_df

,predicate,tp,fp,precision
10,milk,41,0,1.000000
9,hair,39,4,0.906977
4,catsize,32,12,0.727273
13,toothed,40,21,0.655738
5,domestic,8,5,0.615385
3,breathes,41,39,0.512500
2,backbone,41,42,0.493976
12,tail,35,40,0.466667
11,predator,22,34,0.392857
8,fins,4,13,0.235294


In [36]:
best_rule = scores_df.iloc[0]

print("Best predicate:", best_rule["predicate"])
print("True positives:", best_rule["tp"])
print("False positives:", best_rule["fp"])
print("Precision:", best_rule["precision"])

Best predicate: milk
True positives: 41
False positives: 0
Precision: 1.0


In [37]:
best_predicate = best_rule["predicate"]

print(f"mammal(X) :- {best_predicate}(X).")

mammal(X) :- milk(X).


In [38]:
from itertools import combinations

single_predicates = [
    p for p in predicates
    if p != "legs"
]

two_literal_rules = []

for p1, p2 in combinations(single_predicates, 2):
    rule = f"mammal(X) :- {p1}(X), {p2}(X)."
    two_literal_rules.append((p1, p2))

print("Number of two-literal rules:", len(two_literal_rules))

for p1, p2 in two_literal_rules[:20]:
    print(f"mammal(X) :- {p1}(X), {p2}(X).")

Number of two-literal rules: 105
mammal(X) :- airborne(X), aquatic(X).
mammal(X) :- airborne(X), backbone(X).
mammal(X) :- airborne(X), breathes(X).
mammal(X) :- airborne(X), catsize(X).
mammal(X) :- airborne(X), domestic(X).
mammal(X) :- airborne(X), eggs(X).
mammal(X) :- airborne(X), feathers(X).
mammal(X) :- airborne(X), fins(X).
mammal(X) :- airborne(X), hair(X).
mammal(X) :- airborne(X), milk(X).
mammal(X) :- airborne(X), predator(X).
mammal(X) :- airborne(X), tail(X).
mammal(X) :- airborne(X), toothed(X).
mammal(X) :- airborne(X), venomous(X).
mammal(X) :- aquatic(X), backbone(X).
mammal(X) :- aquatic(X), breathes(X).
mammal(X) :- aquatic(X), catsize(X).
mammal(X) :- aquatic(X), domestic(X).
mammal(X) :- aquatic(X), eggs(X).
mammal(X) :- aquatic(X), feathers(X).


In [39]:
def covers_two(animal, p1, p2):
    return (
        covers(animal, p1)
        and
        covers(animal, p2)
    )

In [40]:
print(covers_two("lion", "hair", "milk"))
print(covers_two("lion", "hair", "feathers"))

print(covers_two("chicken", "hair", "milk"))

True
False
False


In [41]:
def evaluate_two_rule(p1, p2):
    true_positives = 0
    false_positives = 0

    # Check positive examples
    for animal in positive_examples:
        if covers_two(animal, p1, p2):
            true_positives += 1

    # Check negative examples
    for animal in negative_examples:
        if covers_two(animal, p1, p2):
            false_positives += 1

    return true_positives, false_positives

In [42]:
def score_two_rule(p1, p2):
    tp, fp = evaluate_two_rule(p1, p2)

    if tp + fp == 0:
        precision = 0
    else:
        precision = tp / (tp + fp)

    return tp, fp, precision

In [43]:
two_rule_scores = []

for p1, p2 in two_literal_rules:

    tp, fp, precision = score_two_rule(p1, p2)

    two_rule_scores.append({
        "p1": p1,
        "p2": p2,
        "tp": tp,
        "fp": fp,
        "precision": precision
    })

two_scores_df = pd.DataFrame(two_rule_scores)

two_scores_df = two_scores_df.sort_values(
    by=["precision", "tp"],
    ascending=[False, False]
)

two_scores_df.head(15)

,p1,p2,tp,fp,precision
34,backbone,milk,41,0,1.0
45,breathes,milk,41,0,1.0
97,milk,toothed,40,0,1.0
33,backbone,hair,39,0,1.0
90,hair,milk,39,0,1.0
93,hair,toothed,38,0,1.0
96,milk,tail,35,0,1.0
92,hair,tail,33,0,1.0
55,catsize,milk,32,0,1.0
54,catsize,hair,30,0,1.0


In [44]:
best_two = two_scores_df.iloc[0]

print("Best two-literal rule:")
print(
    f"mammal(X) :- "
    f"{best_two['p1']}(X), "
    f"{best_two['p2']}(X)."
)

print()
print("True positives :", best_two["tp"])
print("False positives:", best_two["fp"])
print("Precision      :", best_two["precision"])

Best two-literal rule:
mammal(X) :- backbone(X), milk(X).

True positives : 41
False positives: 0
Precision      : 1.0


In [45]:
# ============================================================
# STEP 8 — Handle Predicates with Arguments
# ============================================================

# We want the learner to handle both:
#
#   hair(X)
#   milk(X)
#
# and predicates with arguments:
#
#   legs(X, 4)
#
# This allows rules such as:
#
#   mammal(X) :- hair(X), legs(X,4).
# ============================================================


# ------------------------------------------------------------
# 1. General function for checking whether a fact exists
# ------------------------------------------------------------

def fact_exists(animal, predicate, value=None):

    if value is None:
        fact = f"{predicate}({animal})"
    else:
        fact = f"{predicate}({animal},{value})"

    return fact in background_knowledge[animal]


# ------------------------------------------------------------
# 2. Test the function
# ------------------------------------------------------------

print("Testing fact_exists():")
print("lion has hair:",
      fact_exists("lion", "hair"))

print("lion has 4 legs:",
      fact_exists("lion", "legs", 4))

print("chicken has hair:",
      fact_exists("chicken", "hair"))

print("chicken has 2 legs:",
      fact_exists("chicken", "legs", 2))


# ------------------------------------------------------------
# 3. Find possible values for the legs predicate
# ------------------------------------------------------------

leg_values = sorted(
    df[df["predicate"] == "legs"]["value"]
    .astype(int)
    .unique()
)

print("\nPossible leg values:")
print(leg_values)


# ------------------------------------------------------------
# 4. Generate candidate two-literal rules
#
# Example:
#
# mammal(X) :- hair(X), legs(X,4).
# mammal(X) :- milk(X), legs(X,4).
# mammal(X) :- backbone(X), legs(X,4).
# ------------------------------------------------------------

single_predicates = sorted([
    p for p in df["predicate"].unique()
    if p != "legs"
])

candidate_rules = []

for predicate in single_predicates:

    for leg_value in leg_values:

        candidate_rules.append(
            (predicate, leg_value)
        )


print("\nNumber of candidate rules:",
      len(candidate_rules))

print("\nFirst 15 candidate rules:")

for predicate, leg_value in candidate_rules[:15]:

    print(
        f"mammal(X) :- "
        f"{predicate}(X), "
        f"legs(X,{leg_value})."
    )


# ------------------------------------------------------------
# 5. Check whether an animal satisfies a rule
# ------------------------------------------------------------

def covers_rule(animal, predicate, leg_value):

    return (
        fact_exists(animal, predicate)
        and
        fact_exists(animal, "legs", leg_value)
    )


# ------------------------------------------------------------
# 6. Evaluate a candidate rule
#
# TP = positive mammals correctly covered
# FP = non-mammals incorrectly covered
# ------------------------------------------------------------

def evaluate_rule(predicate, leg_value):

    true_positives = 0
    false_positives = 0

    # Positive examples
    for animal in positive_examples:

        if covers_rule(
            animal,
            predicate,
            leg_value
        ):
            true_positives += 1

    # Negative examples
    for animal in negative_examples:

        if covers_rule(
            animal,
            predicate,
            leg_value
        ):
            false_positives += 1

    return true_positives, false_positives


# ------------------------------------------------------------
# 7. Score all candidate rules
#
# Precision = TP / (TP + FP)
# ------------------------------------------------------------

rule_scores = []

for predicate, leg_value in candidate_rules:

    tp, fp = evaluate_rule(
        predicate,
        leg_value
    )

    if tp + fp == 0:
        precision = 0
    else:
        precision = tp / (tp + fp)

    rule_scores.append({
        "predicate": predicate,
        "leg_value": leg_value,
        "tp": tp,
        "fp": fp,
        "precision": precision
    })


# ------------------------------------------------------------
# 8. Put results into a DataFrame
# ------------------------------------------------------------

rules_df = pd.DataFrame(rule_scores)

rules_df = rules_df.sort_values(
    by=["precision", "tp"],
    ascending=[False, False]
)


# ------------------------------------------------------------
# 9. Display the best rules
# ------------------------------------------------------------

print("\nTop 10 rules:")
print(
    rules_df.head(10).to_string(index=False)
)


# ------------------------------------------------------------
# 10. Display the best rule as First-Order Logic
# ------------------------------------------------------------

best_rule = rules_df.iloc[0]

best_predicate = best_rule["predicate"]
best_leg_value = best_rule["leg_value"]

print("\n" + "=" * 50)
print("BEST LEARNED RULE")
print("=" * 50)

print(
    f"mammal(X) :- "
    f"{best_predicate}(X), "
    f"legs(X,{best_leg_value})."
)

print()
print("True Positives :", best_rule["tp"])
print("False Positives:", best_rule["fp"])
print("Precision      :", round(best_rule["precision"], 4))
print("=" * 50)

Testing fact_exists():
lion has hair: True
lion has 4 legs: True
chicken has hair: False
chicken has 2 legs: True

Possible leg values:
[np.int64(0), np.int64(2), np.int64(4), np.int64(5), np.int64(6), np.int64(8)]

Number of candidate rules: 90

First 15 candidate rules:
mammal(X) :- airborne(X), legs(X,0).
mammal(X) :- airborne(X), legs(X,2).
mammal(X) :- airborne(X), legs(X,4).
mammal(X) :- airborne(X), legs(X,5).
mammal(X) :- airborne(X), legs(X,6).
mammal(X) :- airborne(X), legs(X,8).
mammal(X) :- aquatic(X), legs(X,0).
mammal(X) :- aquatic(X), legs(X,2).
mammal(X) :- aquatic(X), legs(X,4).
mammal(X) :- aquatic(X), legs(X,5).
mammal(X) :- aquatic(X), legs(X,6).
mammal(X) :- aquatic(X), legs(X,8).
mammal(X) :- backbone(X), legs(X,0).
mammal(X) :- backbone(X), legs(X,2).
mammal(X) :- backbone(X), legs(X,4).

Top 10 rules:
predicate  leg_value  tp  fp  precision
     hair          4  31   0   1.000000
     milk          4  31   0   1.000000
 domestic          4   7   0   1.000000
   

In [46]:
# ============================================================
# STEP 9 — Evaluate Rules Using Coverage, Recall and F1
# ============================================================

# Our previous best rule was:
#
# mammal(X) :- hair(X), legs(X,4).
#
# It has:
# TP = 31
# FP = 0
#
# But there are 41 mammals in total.
# Therefore, 10 mammals are not covered.
#
# We now calculate:
#   TP = True Positives
#   FP = False Positives
#   FN = False Negatives
#   TN = True Negatives
#
# and then:
#   Precision
#   Recall
#   F1
# ============================================================


# ------------------------------------------------------------
# 1. Evaluate a rule completely
# ------------------------------------------------------------

def evaluate_complete_rule(predicate, leg_value):

    tp = 0
    fp = 0
    fn = 0
    tn = 0

    # Check positive examples
    for animal in positive_examples:

        if covers_rule(animal, predicate, leg_value):
            tp += 1
        else:
            fn += 1

    # Check negative examples
    for animal in negative_examples:

        if covers_rule(animal, predicate, leg_value):
            fp += 1
        else:
            tn += 1

    return tp, fp, fn, tn


# ------------------------------------------------------------
# 2. Calculate evaluation metrics
# ------------------------------------------------------------

def calculate_metrics(tp, fp, fn, tn):

    # Precision
    if tp + fp > 0:
        precision = tp / (tp + fp)
    else:
        precision = 0

    # Recall
    if tp + fn > 0:
        recall = tp / (tp + fn)
    else:
        recall = 0

    # F1 score
    if precision + recall > 0:
        f1 = (
            2 * precision * recall
            / (precision + recall)
        )
    else:
        f1 = 0

    # Accuracy
    total = tp + fp + fn + tn

    if total > 0:
        accuracy = (tp + tn) / total
    else:
        accuracy = 0

    return precision, recall, f1, accuracy


# ------------------------------------------------------------
# 3. Evaluate every candidate rule
# ------------------------------------------------------------

complete_scores = []

for predicate, leg_value in candidate_rules:

    tp, fp, fn, tn = evaluate_complete_rule(
        predicate,
        leg_value
    )

    precision, recall, f1, accuracy = calculate_metrics(
        tp,
        fp,
        fn,
        tn
    )

    complete_scores.append({
        "predicate": predicate,
        "leg_value": leg_value,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "accuracy": accuracy
    })


# ------------------------------------------------------------
# 4. Create results table
# ------------------------------------------------------------

complete_scores_df = pd.DataFrame(
    complete_scores
)


# ------------------------------------------------------------
# 5. Sort using F1 instead of precision
# ------------------------------------------------------------

complete_scores_df = complete_scores_df.sort_values(
    by=["f1", "recall", "precision"],
    ascending=[False, False, False]
)


# ------------------------------------------------------------
# 6. Display top rules
# ------------------------------------------------------------

print("Top 10 rules based on F1:")
print()

print(
    complete_scores_df[
        [
            "predicate",
            "leg_value",
            "tp",
            "fp",
            "fn",
            "precision",
            "recall",
            "f1"
        ]
    ].head(10).to_string(index=False)
)


# ------------------------------------------------------------
# 7. Get the best rule
# ------------------------------------------------------------

best_rule = complete_scores_df.iloc[0]

print()
print("=" * 60)
print("BEST RULE")
print("=" * 60)

print(
    f"mammal(X) :- "
    f"{best_rule['predicate']}(X), "
    f"legs(X,{best_rule['leg_value']})."
)

print()
print("True Positives :", best_rule["tp"])
print("False Positives:", best_rule["fp"])
print("False Negatives:", best_rule["fn"])
print("True Negatives :", best_rule["tn"])

print()
print("Precision:", round(best_rule["precision"], 4))
print("Recall   :", round(best_rule["recall"], 4))
print("F1       :", round(best_rule["f1"], 4))
print("Accuracy :", round(best_rule["accuracy"], 4))

print("=" * 60)

Top 10 rules based on F1:

predicate  leg_value  tp  fp  fn  precision   recall       f1
     hair          4  31   0  10   1.000000 0.756098 0.861111
     milk          4  31   0  10   1.000000 0.756098 0.861111
 backbone          4  31   6  10   0.837838 0.756098 0.794872
 breathes          4  31   6  10   0.837838 0.756098 0.794872
  toothed          4  30   5  11   0.857143 0.731707 0.789474
     tail          4  28   3  13   0.903226 0.682927 0.777778
  catsize          4  25   1  16   0.961538 0.609756 0.746269
 predator          4  17   5  24   0.772727 0.414634 0.539683
 domestic          4   7   0  34   1.000000 0.170732 0.291667
     hair          2   7   0  34   1.000000 0.170732 0.291667

BEST RULE
mammal(X) :- hair(X), legs(X,4).

True Positives : 31
False Positives: 0
False Negatives: 10
True Negatives : 60

Precision: 1.0
Recall   : 0.7561
F1       : 0.8611
Accuracy : 0.901


In [49]:
# ============================================================
# STEP 10 — Sequential Covering: Learn Multiple Rules
# ============================================================

# Goal:
#
# Instead of finding ONE rule:
#
#     mammal(X) :- hair(X), legs(X,4).
#
# we learn a SET of rules:
#
#     mammal(X) :- hair(X), legs(X,4).
#     mammal(X) :- hair(X), legs(X,2).
#     mammal(X) :- milk(X), legs(X,4).
#
# The rules together form our hypothesis.
# ============================================================


# ------------------------------------------------------------
# 1. Function to find animals covered by a rule
# ------------------------------------------------------------

def covered_animals(predicate, leg_value, examples):

    covered = set()

    for animal in examples:

        if covers_rule(
            animal,
            predicate,
            leg_value
        ):
            covered.add(animal)

    return covered


# ------------------------------------------------------------
# 2. Score a rule against the CURRENT positive examples
# ------------------------------------------------------------

def score_candidate(
    predicate,
    leg_value,
    current_positive,
    all_negative
):

    tp = 0
    fp = 0

    # Current positive examples
    for animal in current_positive:

        if covers_rule(
            animal,
            predicate,
            leg_value
        ):
            tp += 1

    # Negative examples
    for animal in all_negative:

        if covers_rule(
            animal,
            predicate,
            leg_value
        ):
            fp += 1

    if tp + fp > 0:
        precision = tp / (tp + fp)
    else:
        precision = 0

    return tp, fp, precision


# ------------------------------------------------------------
# 3. Learn multiple rules
# ------------------------------------------------------------

remaining_positive = set(positive_examples)

learned_rules = []

# Safety limit so the loop cannot run forever
max_rules = 10


while (
    len(remaining_positive) > 0
    and len(learned_rules) < max_rules
):

    best_rule = None
    best_score = -1

    # Search through all candidate rules
    for predicate, leg_value in candidate_rules:

        tp, fp, precision = score_candidate(
            predicate,
            leg_value,
            remaining_positive,
            negative_examples
        )

        # We want:
        # 1. No false positives
        # 2. High coverage of remaining positives
        # 3. High precision

        if fp == 0 and tp > 0:

            if tp > best_score:

                best_score = tp

                best_rule = {
                    "predicate": predicate,
                    "leg_value": leg_value,
                    "tp": tp,
                    "fp": fp,
                    "precision": precision
                }

    # No useful rule found
    if best_rule is None:
        break

    # Store the rule
    learned_rules.append(best_rule)

    # Find positives covered by this rule
    newly_covered = covered_animals(
        best_rule["predicate"],
        best_rule["leg_value"],
        remaining_positive
    )

    # Remove them from the remaining examples
    remaining_positive -= newly_covered

    # Display progress
    print(
        f"Rule {len(learned_rules)}:"
    )

    print(
        f"  mammal(X) :- "
        f"{best_rule['predicate']}(X), "
        f"legs(X,{best_rule['leg_value']})."
    )


    print()


# ------------------------------------------------------------
# 4. Display the final hypothesis
# ------------------------------------------------------------

print("=" * 60)
print("LEARNED HYPOTHESIS")
print("=" * 60)

for i, rule in enumerate(learned_rules, start=1):

    print(
        f"{i}. mammal(X) :- "
        f"{rule['predicate']}(X), "
        f"legs(X,{rule['leg_value']})."
    )


# ------------------------------------------------------------
# 5. Show remaining positive examples
# ------------------------------------------------------------

print()
print("=" * 60)
print("REMAINING POSITIVE EXAMPLES")
print("=" * 60)

print(
    sorted(remaining_positive)
)

print()
print(
    "Total original positives:",
    len(positive_examples)
)

print(
    "Total covered:",
    len(positive_examples) - len(remaining_positive)
)

print(
    "Total remaining:",
    len(remaining_positive)
)

Rule 1:
  mammal(X) :- hair(X), legs(X,4).

Rule 2:
  mammal(X) :- hair(X), legs(X,2).

Rule 3:
  mammal(X) :- milk(X), legs(X,0).

LEARNED HYPOTHESIS
1. mammal(X) :- hair(X), legs(X,4).
2. mammal(X) :- hair(X), legs(X,2).
3. mammal(X) :- milk(X), legs(X,0).

REMAINING POSITIVE EXAMPLES
[]

Total original positives: 41
Total covered: 41
Total remaining: 0


In [48]:
# STEP 11 — General rule and literal representation

# ---------------------------------------------------------
# 1. Define predicate arities
# ---------------------------------------------------------

predicate_arity = {
    "hair": 1,
    "feathers": 1,
    "eggs": 1,
    "milk": 1,
    "airborne": 1,
    "aquatic": 1,
    "predator": 1,
    "toothed": 1,
    "backbone": 1,
    "breathes": 1,
    "venomous": 1,
    "fins": 1,
    "legs": 2,
    "tail": 1,
    "domestic": 1,
    "catsize": 1
}

# ---------------------------------------------------------
# 2. Represent literals as tuples
# ---------------------------------------------------------

# Unary literal:
# ("hair", "X")
# ("milk", "X")

# Binary literal:
# ("legs", "X", 4)

def make_literal(predicate, value=None):
    """
    Create a generic literal representation.
    """

    arity = predicate_arity[predicate]

    if arity == 1:
        return (predicate, "X")

    elif arity == 2:
        return (predicate, "X", value)

    else:
        raise ValueError("Unsupported predicate arity")


# ---------------------------------------------------------
# 3. Generic literal coverage function
# ---------------------------------------------------------

def covers_literal(animal, literal):
    """
    Check whether an animal satisfies a literal.
    """

    predicate = literal[0]
    arity = predicate_arity[predicate]

    if arity == 1:
        # Example:
        # ("hair", "X")
        fact = f"{predicate}({animal})"

    elif arity == 2:
        # Example:
        # ("legs", "X", 4)
        value = literal[2]
        fact = f"{predicate}({animal},{value})"

    else:
        return False

    return fact in background_knowledge[animal]


# ---------------------------------------------------------
# 4. Generic rule representation
# ---------------------------------------------------------

def make_rule(head_class, body_literals):
    """
    Example:

    make_rule(
        "mammal",
        [
            ("hair", "X"),
            ("legs", "X", 4)
        ]
    )

    represents:

    mammal(X) :- hair(X), legs(X,4).
    """

    return {
        "head": head_class,
        "body": body_literals
    }


# ---------------------------------------------------------
# 5. Generic rule coverage
# ---------------------------------------------------------

def covers_rule_general(animal, rule):
    """
    A rule covers an animal only if ALL
    body literals are satisfied.
    """

    for literal in rule["body"]:
        if not covers_literal(animal, literal):
            return False

    return True


# ---------------------------------------------------------
# 6. Convert rule to readable form
# ---------------------------------------------------------

def rule_to_string(rule):

    head = f'{rule["head"]}(X)'

    body_parts = []

    for literal in rule["body"]:

        predicate = literal[0]
        arity = predicate_arity[predicate]

        if arity == 1:
            body_parts.append(f"{predicate}(X)")

        elif arity == 2:
            value = literal[2]
            body_parts.append(f"{predicate}(X,{value})")

    body = ", ".join(body_parts)

    return f"{head} :- {body}."


# ---------------------------------------------------------
# 7. Generate ALL possible one-literal candidates
# ---------------------------------------------------------

candidate_literals = []

for predicate, arity in predicate_arity.items():

    if arity == 1:
        candidate_literals.append(
            make_literal(predicate)
        )

    elif arity == 2:

        # Get possible values for this predicate
        values = sorted(
            df[df["predicate"] == predicate]["value"]
            .unique()
        )

        for value in values:
            candidate_literals.append(
                make_literal(predicate, value)
            )


# ---------------------------------------------------------
# 8. Create general one-literal rules
# ---------------------------------------------------------

general_rules = []

for literal in candidate_literals:

    rule = make_rule(
        "mammal",
        [literal]
    )

    general_rules.append(rule)


# ---------------------------------------------------------
# 9. Test the general representation
# ---------------------------------------------------------

print("Example literals:")
print(make_literal("hair"))
print(make_literal("milk"))
print(make_literal("legs", 4))

print("\nExample rules:")

for rule in general_rules[:10]:
    print(rule_to_string(rule))


# ---------------------------------------------------------
# 10. Test rule coverage
# ---------------------------------------------------------

test_rule = make_rule(
    "mammal",
    [
        ("hair", "X"),
        ("legs", "X", 4)
    ]
)

print("\nTest rule:")
print(rule_to_string(test_rule))

print("\nCoverage:")

for animal in sorted(list(all_animals))[:10]:

    result = covers_rule_general(
        animal,
        test_rule
    )

    print(animal, "->", result)


# ---------------------------------------------------------
# 11. Evaluate any general rule
# ---------------------------------------------------------

def evaluate_rule_general(rule):

    tp = 0
    fp = 0
    fn = 0
    tn = 0

    for animal in all_animals:

        covered = covers_rule_general(
            animal,
            rule
        )

        is_positive = animal in positive_examples

        if covered and is_positive:
            tp += 1

        elif covered and not is_positive:
            fp += 1

        elif not covered and is_positive:
            fn += 1

        else:
            tn += 1

    precision = (
        tp / (tp + fp)
        if tp + fp > 0
        else 0
    )

    recall = (
        tp / (tp + fn)
        if tp + fn > 0
        else 0
    )

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0
        else 0
    )

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


# ---------------------------------------------------------
# 12. Evaluate all general one-literal rules
# ---------------------------------------------------------

results = []

for rule in general_rules:

    score = evaluate_rule_general(rule)

    results.append({
        "rule": rule_to_string(rule),
        **score
    })


results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="f1",
    ascending=False
).reset_index(drop=True)


print("\nBest general one-literal rules:")
print(
    results_df[
        ["rule", "tp", "fp", "fn", "precision", "recall", "f1"]
    ].head(15).to_string(index=False)
)

Example literals:
('hair', 'X')
('milk', 'X')
('legs', 'X', 4)

Example rules:
mammal(X) :- hair(X).
mammal(X) :- feathers(X).
mammal(X) :- eggs(X).
mammal(X) :- milk(X).
mammal(X) :- airborne(X).
mammal(X) :- aquatic(X).
mammal(X) :- predator(X).
mammal(X) :- toothed(X).
mammal(X) :- backbone(X).
mammal(X) :- breathes(X).

Test rule:
mammal(X) :- hair(X), legs(X,4).

Coverage:
aardvark -> True
antelope -> True
bass -> False
bear -> True
boar -> True
buffalo -> True
calf -> True
carp -> False
catfish -> False
cavy -> True

Best general one-literal rules:
                     rule  tp  fp  fn  precision   recall       f1
    mammal(X) :- milk(X).  41   0   0   1.000000 1.000000 1.000000
    mammal(X) :- hair(X).  39   4   2   0.906977 0.951220 0.928571
  mammal(X) :- legs(X,4).  31   7  10   0.815789 0.756098 0.784810
 mammal(X) :- toothed(X).  40  21   1   0.655738 0.975610 0.784314
 mammal(X) :- catsize(X).  32  12   9   0.727273 0.780488 0.752941
mammal(X) :- breathes(X).  41  39   0

In [50]:
# STEP 12 — Classify Mammal vs Not Mammal using General Rules

# ---------------------------------------------------------
# 1. Positive and negative examples
# ---------------------------------------------------------

positive_examples = {
    animal
    for animal in all_animals
    if animal in df[df["class_name"] == "mammal"]["animal"].values
}

negative_examples = all_animals - positive_examples

print("Mammals:", len(positive_examples))
print("Not mammals:", len(negative_examples))


# ---------------------------------------------------------
# 2. Generate general one-literal rules
# ---------------------------------------------------------

candidate_rules = []

for literal in candidate_literals:

    rule = make_rule(
        "mammal",
        [literal]
    )

    candidate_rules.append(rule)


# ---------------------------------------------------------
# 3. Evaluate a rule
# ---------------------------------------------------------

def evaluate_rule(rule):

    tp = 0
    fp = 0

    for animal in positive_examples:
        if covers_rule_general(animal, rule):
            tp += 1

    for animal in negative_examples:
        if covers_rule_general(animal, rule):
            fp += 1

    precision = (
        tp / (tp + fp)
        if tp + fp > 0
        else 0
    )

    return tp, fp, precision


# ---------------------------------------------------------
# 4. Find the best rules
# ---------------------------------------------------------

rule_results = []

for rule in candidate_rules:

    tp, fp, precision = evaluate_rule(rule)

    if tp > 0:

        rule_results.append({
            "rule": rule,
            "rule_text": rule_to_string(rule),
            "tp": tp,
            "fp": fp,
            "precision": precision
        })


rule_results_df = pd.DataFrame(rule_results)

rule_results_df = rule_results_df.sort_values(
    by=["precision", "tp"],
    ascending=[False, False]
).reset_index(drop=True)


print("\nBest general rules:")
print(
    rule_results_df[
        ["rule_text", "tp", "fp", "precision"]
    ].head(15).to_string(index=False)
)


# ---------------------------------------------------------
# 5. Build a mammal classifier from the best rules
# ---------------------------------------------------------

# Keep rules that make only positive predictions.
# These rules can be combined using OR:
#
# mammal(X) :-
#       rule_1
#       OR rule_2
#       OR rule_3 ...

learned_rules = []

for result in rule_results:

    if result["fp"] == 0:
        learned_rules.append(result["rule"])


# ---------------------------------------------------------
# 6. Remove redundant rules
# ---------------------------------------------------------

# If one rule covers all examples covered by another rule,
# the more specific rule is unnecessary.

def covered_by_rule(rule, examples):

    return {
        animal
        for animal in examples
        if covers_rule_general(animal, rule)
    }


selected_rules = []
covered_positive = set()

for rule in learned_rules:

    newly_covered = (
        covered_by_rule(rule, positive_examples)
        - covered_positive
    )

    if newly_covered:

        selected_rules.append(rule)
        covered_positive.update(newly_covered)


# ---------------------------------------------------------
# 7. Display learned hypothesis
# ---------------------------------------------------------

print("\nLearned Mammal Hypothesis:")
print("-" * 40)

for i, rule in enumerate(selected_rules, start=1):

    print(f"Rule {i}: {rule_to_string(rule)}")


# ---------------------------------------------------------
# 8. Generic classifier
# ---------------------------------------------------------

def classify_animal(animal):

    for rule in selected_rules:

        if covers_rule_general(animal, rule):
            return "mammal"

    return "not mammal"


# ---------------------------------------------------------
# 9. Classify every animal
# ---------------------------------------------------------

classification_results = []

for animal in sorted(all_animals):

    predicted = classify_animal(animal)

    actual = (
        "mammal"
        if animal in positive_examples
        else "not mammal"
    )

    classification_results.append({
        "animal": animal,
        "actual": actual,
        "predicted": predicted,
        "correct": predicted == actual
    })


classification_df = pd.DataFrame(
    classification_results
)


# ---------------------------------------------------------
# 10. Show classification results
# ---------------------------------------------------------

print("\nClassification Results:")
print(
    classification_df.to_string(index=False)
)


# ---------------------------------------------------------
# 11. Calculate final classifier performance
# ---------------------------------------------------------

tp = 0
tn = 0
fp = 0
fn = 0

for _, row in classification_df.iterrows():

    actual = row["actual"]
    predicted = row["predicted"]

    if actual == "mammal" and predicted == "mammal":
        tp += 1

    elif actual == "not mammal" and predicted == "not mammal":
        tn += 1

    elif actual == "not mammal" and predicted == "mammal":
        fp += 1

    elif actual == "mammal" and predicted == "not mammal":
        fn += 1


precision = (
    tp / (tp + fp)
    if tp + fp > 0
    else 0
)

recall = (
    tp / (tp + fn)
    if tp + fn > 0
    else 0
)

accuracy = (
    (tp + tn) / (tp + tn + fp + fn)
    if tp + tn + fp + fn > 0
    else 0
)

f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall > 0
    else 0
)


print("\nFinal Performance")
print("-" * 40)
print("TP       :", tp)
print("TN       :", tn)
print("FP       :", fp)
print("FN       :", fn)
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1 Score :", round(f1, 4))
print("Accuracy :", round(accuracy, 4))

Mammals: 41
Not mammals: 60

Best general rules:
                rule_text  tp  fp  precision
    mammal(X) :- milk(X).  41   0   1.000000
    mammal(X) :- hair(X).  39   4   0.906977
  mammal(X) :- legs(X,4).  31   7   0.815789
 mammal(X) :- catsize(X).  32  12   0.727273
 mammal(X) :- toothed(X).  40  21   0.655738
mammal(X) :- domestic(X).   8   5   0.615385
mammal(X) :- breathes(X).  41  39   0.512500
mammal(X) :- backbone(X).  41  42   0.493976
    mammal(X) :- tail(X).  35  40   0.466667
mammal(X) :- predator(X).  22  34   0.392857
  mammal(X) :- legs(X,2).   7  20   0.259259
    mammal(X) :- fins(X).   4  13   0.235294
 mammal(X) :- aquatic(X).   6  30   0.166667
  mammal(X) :- legs(X,0).   3  20   0.130435
mammal(X) :- airborne(X).   2  22   0.083333

Learned Mammal Hypothesis:
----------------------------------------
Rule 1: mammal(X) :- milk(X).

Classification Results:
  animal     actual  predicted  correct
aardvark     mammal     mammal     True
antelope     mammal     mam

In [51]:
# Print only the learned mammal rules

print("Learned Rules:")
print("-" * 40)

for rule in selected_rules:
    print(rule_to_string(rule))

Learned Rules:
----------------------------------------
mammal(X) :- milk(X).


In [52]:
# Print ALL rules in descending order of quality

rule_results_df = rule_results_df.sort_values(
    by=["precision", "tp"],
    ascending=[False, False]
).reset_index(drop=True)

print("All Rules (Descending Order)")
print("-" * 60)

for _, row in rule_results_df.iterrows():
    print(
        f"{row['rule_text']}  "
        f"[TP={row['tp']}, FP={row['fp']}, "
        f"Precision={row['precision']:.4f}]"
    )

All Rules (Descending Order)
------------------------------------------------------------
mammal(X) :- milk(X).  [TP=41, FP=0, Precision=1.0000]
mammal(X) :- hair(X).  [TP=39, FP=4, Precision=0.9070]
mammal(X) :- legs(X,4).  [TP=31, FP=7, Precision=0.8158]
mammal(X) :- catsize(X).  [TP=32, FP=12, Precision=0.7273]
mammal(X) :- toothed(X).  [TP=40, FP=21, Precision=0.6557]
mammal(X) :- domestic(X).  [TP=8, FP=5, Precision=0.6154]
mammal(X) :- breathes(X).  [TP=41, FP=39, Precision=0.5125]
mammal(X) :- backbone(X).  [TP=41, FP=42, Precision=0.4940]
mammal(X) :- tail(X).  [TP=35, FP=40, Precision=0.4667]
mammal(X) :- predator(X).  [TP=22, FP=34, Precision=0.3929]
mammal(X) :- legs(X,2).  [TP=7, FP=20, Precision=0.2593]
mammal(X) :- fins(X).  [TP=4, FP=13, Precision=0.2353]
mammal(X) :- aquatic(X).  [TP=6, FP=30, Precision=0.1667]
mammal(X) :- legs(X,0).  [TP=3, FP=20, Precision=0.1304]
mammal(X) :- airborne(X).  [TP=2, FP=22, Precision=0.0833]
mammal(X) :- eggs(X).  [TP=1, FP=58, Precisio

In [53]:
# Use multiple rules to classify mammal vs not mammal

# ---------------------------------------------------------
# 1. Select multiple good rules
# ---------------------------------------------------------

# Rules must have no false positives.
# Sort them by the number of positive examples they cover.

good_rules = [
    result["rule"]
    for result in rule_results
    if result["fp"] == 0 and result["tp"] > 0
]

good_rules = sorted(
    good_rules,
    key=lambda rule: evaluate_rule(rule)[0],
    reverse=True
)


# ---------------------------------------------------------
# 2. Combine multiple rules
# ---------------------------------------------------------

selected_rules = []
covered_positive = set()

for rule in good_rules:

    newly_covered = {
        animal
        for animal in positive_examples
        if covers_rule_general(animal, rule)
    }

    # Only keep the rule if it covers at least
    # one mammal not already covered.
    if newly_covered - covered_positive:

        selected_rules.append(rule)
        covered_positive.update(newly_covered)

    # Stop when every mammal is covered
    if covered_positive == positive_examples:
        break


# ---------------------------------------------------------
# 3. Print the multiple-rule hypothesis
# ---------------------------------------------------------

print("Learned Multiple-Rule Hypothesis")
print("-" * 50)

for i, rule in enumerate(selected_rules, 1):
    print(f"Rule {i}: {rule_to_string(rule)}")


# ---------------------------------------------------------
# 4. Classifier using all rules
# ---------------------------------------------------------

def classify_animal(animal):

    # OR between rules:
    # If ANY rule covers the animal -> mammal

    for rule in selected_rules:

        if covers_rule_general(animal, rule):
            return "mammal"

    return "not mammal"


# ---------------------------------------------------------
# 5. Show only the final rules
# ---------------------------------------------------------

print("\nFinal Rules:")
for rule in selected_rules:
    print(rule_to_string(rule))


Learned Multiple-Rule Hypothesis
--------------------------------------------------
Rule 1: mammal(X) :- milk(X).

Final Rules:
mammal(X) :- milk(X).
